In [3]:
import pandas as pd
import numpy as np
import json

X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = np.load('data/processed/y_train.npy')
y_test = np.load('data/processed/y_test.npy')

with open('data/processed/label_mapping.json') as f:
    label_mapping = json.load(f)

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_test_scaled = scaler.transform(X_test.values)

In [5]:
X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_reshaped = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

In [6]:
import tensorflow as tf
y_train_onehot = tf.keras.utils.to_categorical(y_train)
y_test_onehot = tf.keras.utils.to_categorical(y_test)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

num_classes = y_train_onehot.shape[1]
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(23, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

C:\Users\silpi\anaconda3\envs\traffic-classifier\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
from tensorflow.keras.callbacks import EarlyStopping

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)

history = model.fit(
    X_train_reshaped, y_train_onehot,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5534 - loss: 1.2494 - val_accuracy: 0.5723 - val_loss: 1.1707
Epoch 2/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5596 - loss: 1.2386 - val_accuracy: 0.5851 - val_loss: 1.1758
Epoch 3/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5630 - loss: 1.2333 - val_accuracy: 0.5859 - val_loss: 1.1694
Epoch 4/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5651 - loss: 1.2205 - val_accuracy: 0.5945 - val_loss: 1.1577
Epoch 5/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5687 - loss: 1.2163 - val_accuracy: 0.6084 - val_loss: 1.1360
Epoch 6/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5708 - loss: 1.2039 - val_accuracy: 0.5746 - val_loss: 1.1676
Epoch 7/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5730 - loss: 1.1974 - val_accuracy: 0.5859 - val_loss: 1.1452
Epoch 8/50
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.5726 - loss: 1.1919 - 

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

y_pred_probs = model.predict(X_test_reshaped)
y_pred = np.argmax(y_pred_probs, axis=1)

id_to_label = {v: k for k, v in label_mapping.items()}
target_names = [id_to_label[i] for i in sorted(id_to_label.keys())]

accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}\n")

print(classification_report(y_test, y_pred, target_names=target_names))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

374/374 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Test Accuracy: 0.6376

               precision    recall  f1-score   support

     BROWSING       0.63      0.81      0.71      2000
         CHAT       0.56      0.44      0.49       501
           FT       0.73      0.54      0.62       795
         MAIL       0.62      0.18      0.28       273
          P2P       0.55      0.80      0.65       800
    STREAMING       0.70      0.40      0.51       257
         VOIP       0.74      0.91      0.82      1297
 VPN-BROWSING       0.61      0.63      0.62      2000
     VPN-CHAT       0.58      0.27      0.37       568
       VPN-FT       0.72      0.43      0.54       941
     VPN-MAIL       0.57      0.77      0.65       489
      VPN-P2P       0.48      0.64      0.54       683
VPN-STREAMING       0.75      0.77      0.76       223
     VPN-VOIP       0.83      0.49      0.62      1115

     accuracy                           0.64     11942
    macro avg       0.65      0.58      0.58     11942

In [12]:
model.save('models/cnn_model.h5')
print("Model saved")

Model saved
